# Cloud Service Providers — Data Cleaning Pipeline

This notebook runs all 20 cleaning scenarios sequentially on the raw `usage_billing.csv` dataset.
Each scenario has:
- **Cleaning cell** — calls the scenario `.py` file from `scenarios/`
- **Validation cell** — calls the validation `.py` file from `validations/`

At the end, we assemble the final cleaned dataset and export it.

In [ ]:
import pandas as pd
import numpy as np
import sys
import warnings

warnings.filterwarnings('ignore')
sys.path.insert(0, '.')

df = pd.read_csv('data/raw/usage_billing.csv')
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.sample(5, random_state=42)

---
# S01 — Account ID Normalization & Master Mapping

**Output columns:** `Account_Clean`, `Account_In_Master`

In [ ]:
from scenarios.s01_account_id import run as s01_run
df = s01_run(df)
changed = df[df['Account_ID'] != df['Account_Clean']]
print(f"\nSample cleaned rows:")
print(changed[['Account_ID', 'Account_Clean', 'Account_In_Master']].head(10).to_string(index=False))

In [ ]:
from scenarios.s01_account_id import load_master_accounts
from validations.v01_account_id import validate as v01_validate
master_accounts, _ = load_master_accounts()
passed, failed, _ = v01_validate(df, master_accounts)

---
# S02 — Timestamp Normalization to UTC

**Output columns:** `TS_UTC`, `TS_Parse_Failed`, `TS_Garbage_Flag`

In [ ]:
from scenarios.s02_timestamp import run as s02_run
df = s02_run(df)
print(f"\nSample parsed timestamps:")
print(df[['Timestamp', 'TS_UTC', 'TS_Parse_Failed', 'TS_Garbage_Flag']].head(10).to_string(index=False))

In [ ]:
from validations.v02_timestamp import validate as v02_validate
passed, failed, _ = v02_validate(df)

---
# S03 — SKU & Service Normalization

**Output columns:** `SKU_Clean`, `SKU_Unmatched`, `SKU_Changed`, `Service_Clean`

In [ ]:
from scenarios.s03_sku import run as s03_run
df = s03_run(df)
changed = df[df['SKU_Changed']]
print(f"\nSample cleaned SKUs:")
print(changed[['SKU', 'SKU_Clean', 'Service', 'Service_Clean']].head(10).to_string(index=False))

In [ ]:
from validations.v03_sku import validate as v03_validate
passed, failed, _ = v03_validate(df)

---
# S04 — Unit Normalization & Value Conversion

**Output columns:** `Unit_Canonical`, `Usage_Converted`, `Unit_Dimension_Mismatch`

In [ ]:
from scenarios.s04_unit import run as s04_run
df = s04_run(df)
print(f"\nSample unit conversions:")
print(df[['Unit', 'Unit_Canonical', 'Usage_Value', 'Usage_Converted', 'Unit_Dimension_Mismatch']].sample(10, random_state=42).to_string(index=False))

In [ ]:
from validations.v04_unit import validate as v04_validate
passed, failed, _ = v04_validate(df)

---
# S05 — Cost Cleaning & Currency Normalization

**Output columns:** `Cost_Clean`, `Currency_Clean`, `Is_Negative_Cost`, `Is_Zero_Cost`

In [ ]:
from scenarios.s05_cost import run as s05_run
df = s05_run(df)
print(f"\nSample cost cleaning:")
print(df[['Cost', 'Cost_Clean', 'Currency', 'Currency_Clean', 'Is_Negative_Cost', 'Is_Zero_Cost']].sample(10, random_state=42).to_string(index=False))

In [ ]:
from validations.v05_cost import validate as v05_validate
passed, failed, _ = v05_validate(df)

---
# S06 — Region Normalization

**Output columns:** `Region_Clean`, `Region_Unresolvable`

In [ ]:
from scenarios.s06_region import run as s06_run
df = s06_run(df)
changed = df[df['Region'] != df['Region_Clean']]
print(f"\nSample region cleaning:")
print(changed[['Region', 'Region_Clean', 'Region_Unresolvable']].head(10).to_string(index=False))

In [ ]:
from validations.v06_region import validate as v06_validate
passed, failed, _ = v06_validate(df)

---
# S07 — Duplicate Detection

**Problem:** ~550 exact duplicate rows injected. Need to identify all copies and mark first occurrence to keep.

**Solution:** Hash-based duplicate detection on raw columns. Flag all duplicates, mark first-to-keep.

**Output columns:** `Is_Duplicate` (bool), `Duplicate_Keep` (bool — True for first occurrence)

In [ ]:
from scenarios.s07_duplicates import run as s07_run
df = s07_run(df)
dups = df[df['Is_Duplicate']].sort_values('Usage_ID')
print(f"\nSample duplicates:")
print(dups[['Usage_ID', 'Account_ID', 'SKU', 'Is_Duplicate', 'Duplicate_Keep']].head(10).to_string(index=False))

In [ ]:
from validations.v07_duplicates import validate as v07_validate
passed, failed, _ = v07_validate(df)

---
# S08–S20 — Coming Next

Scenarios will be added here as they are built.